# 04 · Predicciones de movilidad turística en Petén

**Objetivo.** Estimar enero y febrero de 2026, meses sin cobertura telefónica, utilizando estacionalidad de migración nacional y el factor de reparto observado en Petén. Las cifras son **estimaciones**, no observaciones.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.predicciones import cargar, perfil_historico, estimar_nacional, estimar_destino, validar_retroactivo
DATA = ROOT / "data" / "processed"

## 1. Fuentes y cobertura
La serie nacional sirve como patrón de estacionalidad; la telefonía mide movilidad estimada en polígonos de Petén. No son el mismo universo. No se suman polígonos para obtener un total único del departamento.

In [2]:
mov, mig = cargar(DATA)
print("Telefonía:", mov.fecha.min().date(), "a", mov.fecha.max().date())
print("Migración:", mig.fecha.min().date(), "a", mig.fecha.max().date())
display(mov.groupby([mov.fecha.dt.strftime("%Y-%m"),"subdestino"]).size().unstack(fill_value=0))

Telefonía: 2025-09-01 a 2026-08-01
Migración: 2009-01-01 a 2026-06-01


subdestino,El Remate Y Alrededores Lago,Isla De Flores,Mundo Maya Aeropuerto
fecha,,,
2025-09,2,2,2
2025-10,2,2,2
2025-11,2,2,2
2025-12,2,2,2
2026-03,2,2,2
2026-04,2,2,2
2026-05,2,2,2
2026-06,2,2,2
2026-07,2,2,2


## 2. Perfil estacional nacional
Se promedian los índices mensuales normalizados dentro de cada año completo 2017–2024. Se excluye 2025 del perfil histórico para usarlo como año de calibración.

In [3]:
perfil = perfil_historico(mig)
display(perfil)
px.line(perfil,x="mes",y="indice",markers=True,title="Perfil estacional nacional (promedio = 1)").show()

,mes,indice
0,1,1.383659
1,2,1.154409
2,3,1.122987
3,4,0.925646
4,5,0.769209
5,6,0.843085
6,7,0.932457
7,8,0.946687
8,9,0.753532
9,10,0.813042


## 3. Estimación nacional y reparto territorial
Se estima el nivel nacional de 2026 usando la mediana de migración observada / índice estacional de septiembre–diciembre 2025. Luego se aplica el factor mediano de telefonía / migración nacional observado en los mismos cuatro meses, por subdestino y segmento. **No se utilizan datos observados de enero–febrero 2026 para calibrar.**

In [4]:
nacional = estimar_nacional(mig, perfil, [1,2])
pred, factores = estimar_destino(mov,mig,nacional)
display(nacional)
display(factores)
display(pred.head(12))

,fecha,migracion_nacional_estimada
0,2026-01-01,443830.835569
1,2026-02-01,370295.187308


,subdestino,segmento,factor_reparto,meses_calibracion
0,El Remate Y Alrededores Lago,mayor a 1 dia,0.106461,4
1,El Remate Y Alrededores Lago,menor a 1 dia,0.198543,4
2,Isla De Flores,mayor a 1 dia,0.122090,4
3,Isla De Flores,menor a 1 dia,0.197820,4
4,Mundo Maya Aeropuerto,mayor a 1 dia,0.136383,4
5,Mundo Maya Aeropuerto,menor a 1 dia,0.227289,4


,fecha,subdestino,segmento,visitantes_estimados,tipo_dato,metodo,factor_reparto,meses_calibracion,migracion_nacional_estimada
0,2026-01-01,El Remate Y Alrededores Lago,mayor a 1 dia,47250.587377,Estimación,Perfil histórico nacional + factor mediano sep...,0.106461,4,443830.835569
1,2026-01-01,El Remate Y Alrededores Lago,menor a 1 dia,88119.476518,Estimación,Perfil histórico nacional + factor mediano sep...,0.198543,4,443830.835569
2,2026-01-01,Isla De Flores,mayor a 1 dia,54187.149972,Estimación,Perfil histórico nacional + factor mediano sep...,0.122090,4,443830.835569
3,2026-01-01,Isla De Flores,menor a 1 dia,87798.590204,Estimación,Perfil histórico nacional + factor mediano sep...,0.197820,4,443830.835569
4,2026-01-01,Mundo Maya Aeropuerto,mayor a 1 dia,60530.963285,Estimación,Perfil histórico nacional + factor mediano sep...,0.136383,4,443830.835569
5,2026-01-01,Mundo Maya Aeropuerto,menor a 1 dia,100877.803564,Estimación,Perfil histórico nacional + factor mediano sep...,0.227289,4,443830.835569
6,2026-02-01,El Remate Y Alrededores Lago,mayor a 1 dia,39421.923176,Estimación,Perfil histórico nacional + factor mediano sep...,0.106461,4,370295.187308
7,2026-02-01,El Remate Y Alrededores Lago,menor a 1 dia,73519.493121,Estimación,Perfil histórico nacional + factor mediano sep...,0.198543,4,370295.187308
8,2026-02-01,Isla De Flores,mayor a 1 dia,45209.208645,Estimación,Perfil histórico nacional + factor mediano sep...,0.122090,4,370295.187308
9,2026-02-01,Isla De Flores,menor a 1 dia,73251.772522,Estimación,Perfil histórico nacional + factor mediano sep...,0.197820,4,370295.187308


In [5]:
px.bar(pred,x="fecha",y="visitantes_estimados",color="segmento",facet_col="subdestino",barmode="group",title="Estimación ene–feb 2026 por polígono (no sumar polígonos)").show()

## 4. Validación retrospectiva
Se deja fuera, uno a uno, cada mes de septiembre–diciembre 2025. El factor se calcula con los otros tres meses y se predice el mes excluido. Se reportan MAE y RMSE; **esto valida el factor de reparto en meses observados, no garantiza exactitud de la extrapolación a enero–febrero 2026**.

In [6]:
detalle, metricas = validar_retroactivo(mov,mig)
display(metricas)
display(detalle.head(12))

,subdestino,segmento,MAE,RMSE,meses_evaluados
0,El Remate Y Alrededores Lago,mayor a 1 dia,3494.384972,3567.612894,4
1,El Remate Y Alrededores Lago,menor a 1 dia,12720.543774,12978.756218,4
2,Isla De Flores,mayor a 1 dia,6580.750584,6854.141742,4
3,Isla De Flores,menor a 1 dia,9288.132687,9538.358984,4
4,Mundo Maya Aeropuerto,mayor a 1 dia,3081.007568,3290.412653,4
5,Mundo Maya Aeropuerto,menor a 1 dia,7912.461677,8252.347960,4


,fecha,subdestino,segmento,observado,estimado,error
0,2025-09-01,El Remate Y Alrededores Lago,mayor a 1 dia,28469.221902,24077.383699,-4391.838203
1,2025-09-01,El Remate Y Alrededores Lago,menor a 1 dia,52499.726681,42295.565690,-10204.160992
2,2025-09-01,Isla De Flores,mayor a 1 dia,34594.799822,27007.793684,-7587.006139
3,2025-09-01,Isla De Flores,menor a 1 dia,55429.961276,43983.749383,-11446.211893
4,2025-09-01,Mundo Maya Aeropuerto,mayor a 1 dia,36358.438642,31420.730191,-4937.708451
5,2025-09-01,Mundo Maya Aeropuerto,menor a 1 dia,63012.202145,51647.768765,-11364.433380
6,2025-10-01,El Remate Y Alrededores Lago,mayor a 1 dia,30808.712989,28234.235715,-2574.477273
7,2025-10-01,El Remate Y Alrededores Lago,menor a 1 dia,60513.809494,49597.704897,-10916.104596
8,2025-10-01,Isla De Flores,mayor a 1 dia,36040.106153,31670.567806,-4369.538347
9,2025-10-01,Isla De Flores,menor a 1 dia,58133.198347,51577.345914,-6555.852433


In [7]:
px.scatter(detalle,x="observado",y="estimado",color="subdestino",symbol="segmento",title="Validación retrospectiva: observado vs estimado").show()

## 5. Exportación reproducible
El script `python -m src.predicciones` genera seis CSV en `data/processed/predicciones/`. Los datos originales no se modifican.

**Limitaciones:** solo cuatro meses para calibración, variabilidad entre temporadas, posibles diferencias entre movilidad y migración, y ausencia de datos telefónicos observados en enero–febrero 2026. No interpretar las estimaciones como un total único de Petén.

In [8]:
from src.predicciones import ejecutar
salidas = ejecutar(DATA, DATA / "predicciones")
print("Archivos generados:", len(salidas))

perfil_estacional_nacional.csv: 12 filas
migracion_nacional_estimada_2026.csv: 2 filas
factores_reparto.csv: 6 filas
prediccion_peten_ene_feb_2026.csv: 12 filas
validacion_retroactiva.csv: 24 filas
metricas_prediccion.csv: 6 filas
Archivos generados: 6
